# TDI — run, then validate

Local notebook for `D:\CYP450\cyp`. Trains the classifier pool once, then spends the rest
of its cells asking whether the result deserves a submission slot. Run top to bottom the
first time; after that, Cell 3 skips retraining and every analysis cell is seconds.

## The two endpoints need opposite work — this is the fact the whole notebook is built on

Inverting every competitor's published accuracy / precision / recall recovers their cut
rate, the blind prevalence and their discriminability `d'`. Prevalence came out to 0.2915
(CYP3A4) and 0.0706 (CYP2D6) on **every** competitor to four decimals, which is what
makes the inversion trustworthy rather than a story.

| | our d' | best competitor | our cut | their cut | diagnosis |
|---|---|---|---|---|---|
| **CYP3A4** | 1.290 | 1.319 (jeremy), 1.289 (wbot) | 0.229 | 0.290–0.314 | model is **level with the field**; the cut is too tight |
| **CYP2D6** | 0.980 | 1.392 (jeremy) | 0.214 | 0.082 | cut is nearly right; the gap is **discriminability** |

So: **judge CYP3A4 work on the threshold, judge CYP2D6 work on AUC.** An improvement to
CYP3A4's AUC is worth little; an improvement to CYP2D6's threshold is worth nothing.

## What this notebook exists to prevent

`submission_tdi_recut` went 0.3325 → 0.3187 (rank 15 → 18). The threshold half of that
change worked exactly as predicted (+0.0087). The other half swapped in a different
score column on a 97.9% label-agreement argument, and the 53 labels that *did* change
sat on the decision boundary, costing −0.0358. **Cell 19 is the diff that would have
caught it**, and the `tdiI` scores it needed are permanently lost, so run it every time.

## Standing rule

> We do not need to submit little modifications — the daily cap means a submission
> should be a big jump. Calibration is a last resort; the model comes first.

## Cell 1 — Setup

Points at the repo and imports the project's own modules. Nothing here is re-implemented: every quantity the pipeline computes is imported from the code that computes it, because the one time this project re-implemented a library's scaffold split it silently contaminated 78% of its own out-of-fold rows.

In [ ]:
import os, sys, time, subprocess, itertools, warnings
from pathlib import Path

REPO = Path(r'D:\CYP450\cyp')          # <-- edit if the repo lives elsewhere
os.chdir(REPO); sys.path.insert(0, str(REPO))
warnings.filterwarnings('ignore')

import numpy as np, pandas as pd
from scipy.stats import norm, spearmanr
from sklearn.metrics import roc_auc_score, matthews_corrcoef

from cypkit import config as CC
from cypkit import tdi as T
from scripts.run_tdi_ensemble import (
    POOL, OPTIMAL_RATE, RATE_INFLATION, TEST_PREVALENCE_BY_ISO, submitted_rate)

TAG      = 'tdiNB'                     # artifacts get this suffix
ISOS     = list(CC.TDI_ISOFORMS)       # ('CYP3A4', 'CYP2D6')
ART      = REPO / 'artifacts'
BASELINE = ART / 'submission_tdi_best.csv'   # the standing best, MA-MCC 0.3325, rank 15
BEST_MCC = {'CYP3A4': 0.4470, 'CYP2D6': 0.2180}
BEST_MA  = float(np.mean(list(BEST_MCC.values())))

pd.set_option('display.width', 160, 'display.max_columns', 40)
print('repo      :', REPO)
print('endpoints :', ISOS)
print('pool      :', list(POOL))
print('prevalence:', TEST_PREVALENCE_BY_ISO)
print('optimal effective cut rate:', OPTIMAL_RATE)
print('rate inflation (submitted -> effective):',
      {k: round(v, 4) for k, v in RATE_INFLATION.items()})
print(f'\nstanding best: MA-MCC {BEST_MA:.4f}  {BEST_MCC}')

## Cell 2–3 — Train the pool

Runs `scripts/run_tdi_ensemble.py --save-oof --predict`, which is the same entry point
the CLI uses, so the notebook cannot drift from it.

`--save-oof` is what makes everything below possible: it persists **every member's**
scores for both the training folds and the blind set. Without the blind half, a weight
refit can be measured but never submitted — which is exactly what blocked this project
once already.

Set `FORCE = True` to retrain. Expect a few minutes on CPU.

Each line is stamped with elapsed seconds, and the child is launched with `-u` plus
`PYTHONUNBUFFERED=1`. Without those the cell prints the command and then goes silent for
the entire run — Python block-buffers stdout when it is piped rather than attached to a
terminal, and `Popen(bufsize=1)` only governs the reading end. Silence there is buffering,
not a hang.

In [ ]:
FORCE = False

need = [ART / f'tdi_member_oof_{c}_{TAG}.csv' for c in ISOS] + \
       [ART / f'testscores_{TAG}.csv']
missing = [p for p in need if not p.exists()]

if FORCE or missing:
    if missing:
        print('missing:', [p.name for p in missing], '\n')
    # -u and PYTHONUNBUFFERED both matter. Popen's bufsize only controls OUR end of the
    # pipe; the CHILD still block-buffers its stdout when it is not a terminal, so
    # without these the cell prints the command and then sits in apparent silence for
    # the whole run. That looks exactly like a hang and it is not one.
    cmd = [sys.executable, '-u', 'scripts/run_tdi_ensemble.py',
           '--save-oof', '--predict', '--tag', TAG]
    env = dict(os.environ, PYTHONUNBUFFERED='1')
    print(' '.join(cmd), '\n' + '-' * 70, flush=True)
    t0 = time.time()
    p = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                         text=True, bufsize=1, cwd=str(REPO), env=env)
    for line in p.stdout:
        print(f'[{time.time()-t0:6.1f}s] {line}', end='', flush=True)
    rc = p.wait()
    print(f'\nfinished in {(time.time()-t0)/60:.1f} min', flush=True)
    if rc != 0:
        raise SystemExit(f'run_tdi_ensemble.py exited {rc} -- fix that before continuing')
else:
    print('artifacts present; skipping the retrain. Set FORCE = True to redo it.')
    for p in need:
        print(f'  {p.name}')

## Cell 4–5 — Load and sanity-check

Four things are checked before any number is believed:

1. **every member covers every compound** — a member with NaNs quietly drops out of a
   rank average and its weight silently moves to the others;
2. **the label is not degenerate** — MCC is undefined on a constant column;
3. **out-of-fold prevalence vs blind prevalence** — these are different populations and
   the gap is the reason a good OOF MCC does not transfer;
4. **the blind file covers all 750 compounds.**

In [ ]:
oof, test_member = {}, {}
for c in ISOS:
    oof[c] = pd.read_csv(ART / f'tdi_member_oof_{c}_{TAG}.csv')

MEMBERS = [m for m in oof[ISOS[0]].columns if m not in (CC.ID_COL, 'y')]
for m in MEMBERS:
    f = ART / f'tdi_member_test_{m}_{TAG}.csv'
    if f.exists():
        test_member[m] = pd.read_csv(f)
blend_test = pd.read_csv(ART / f'testscores_{TAG}.csv')

print(f'members ({len(MEMBERS)}): {MEMBERS}')
print(f'per-member blind score files found: {len(test_member)}\n')

ok = True
for c in ISOS:
    d = oof[c]; y = d['y'].to_numpy()
    nan = {m: int(d[m].isna().sum()) for m in MEMBERS if d[m].isna().any()}
    print(f'{c}: {len(d)} labelled rows, positives {int(y.sum())} '
          f'({y.mean()*100:.1f}%)   blind prevalence {TEST_PREVALENCE_BY_ISO[c]*100:.1f}%')
    if nan:
        print(f'   !! NaN scores: {nan} -- these members silently lose weight'); ok = False
    if y.min() == y.max():
        print('   !! label is constant'); ok = False
    r = TEST_PREVALENCE_BY_ISO[c] / y.mean()
    print(f'   blind/OOF prevalence ratio {r:.2f}'
          f'{"   <- populations differ; OOF MCC will not transfer directly" if abs(r-1) > 0.15 else ""}')

print(f'\nblind rows: {len(blend_test)} (expect 750)')
assert len(blend_test) == 750, 'blind file does not cover the test set'
print('sanity checks', 'PASSED' if ok else 'RAISED WARNINGS -- read them before continuing')

## Cell 6–7 — The honest per-member scoreboard

AUC and best-achievable MCC per member per endpoint.

Read **AUC**, not MCC. `best_mcc_threshold` searches the threshold on the same data it
scores, so every MCC here is optimistically biased by a different amount per member —
it is a ceiling, not an estimate. AUC is threshold-free and comparable across members.

In [ ]:
rows = []
for m in MEMBERS + ['RANK-BLEND']:
    r = {'member': m}
    for c in ISOS:
        d = oof[c]; y = d['y'].to_numpy().astype(bool)
        if m == 'RANK-BLEND':
            R = np.vstack([pd.Series(d[x]).rank().to_numpy() for x in MEMBERS])
            s = np.nanmean(R, axis=0)
        else:
            s = d[m].to_numpy(float)
        g = np.isfinite(s)
        r[f'AUC_{c}'] = roc_auc_score(y[g], s[g])
        r[f'MCC_{c}'] = T.best_mcc_threshold(y[g], s[g])[1]
    r['AUC_macro'] = np.mean([r[f'AUC_{c}'] for c in ISOS])
    rows.append(r)

board = pd.DataFrame(rows).sort_values('AUC_macro', ascending=False)
print(board.to_string(index=False, float_format=lambda x: f'{x:.4f}'))
print('\nBlind-set AUC implied by our scored submission: CYP3A4 0.820, CYP2D6 0.756.')
print('An OOF AUC far above those is measuring the wrong population, not a better model.')

## Cell 8–9 — Diversity

Rank-correlation between members. Two members correlating above ~0.95 are one member
counted twice, and in a rank average that silently doubles its weight.

This is also where a *new* member should be judged first: a candidate that agrees with
`lgb_clf` at 0.97 cannot help however good it looks on its own.

In [ ]:
for c in ISOS:
    d = oof[c]
    R = d[MEMBERS].rank().corr(method='pearson')   # Pearson of ranks == Spearman
    print(f'--- {c} ---')
    print(R.round(3).to_string())
    hi = [(a, b, R.loc[a, b]) for a, b in itertools.combinations(MEMBERS, 2)
          if R.loc[a, b] > 0.95]
    print('redundant pairs (>0.95):', hi if hi else 'none')
    print()

## Cell 10–11 — Ablation: which members are actually carrying the blend?

Drop each member from the rank average and measure the change in AUC. A member with a
positive delta when removed is **costing** you.

Caveat worth keeping in view: this is leave-one-out on out-of-fold scores, so it is
itself a fitted quantity. Treat a delta smaller than about 0.005 as noise, and prefer
dropping a member only when the ablation and the correlation matrix agree.

In [ ]:
def blend(d, members):
    R = np.vstack([pd.Series(d[m]).rank().to_numpy() for m in members])
    return np.nanmean(R, axis=0)

rows = []
for c in ISOS:
    d = oof[c]; y = d['y'].to_numpy().astype(bool)
    full = roc_auc_score(y, blend(d, MEMBERS))
    for m in MEMBERS:
        rest = [x for x in MEMBERS if x != m]
        rows.append({'endpoint': c, 'dropped': m,
                     'AUC_without': roc_auc_score(y, blend(d, rest)),
                     'delta': roc_auc_score(y, blend(d, rest)) - full})
    rows.append({'endpoint': c, 'dropped': '(none)', 'AUC_without': full, 'delta': 0.0})

ab = pd.DataFrame(rows).pivot(index='dropped', columns='endpoint', values='delta')
print('change in blend AUC when a member is REMOVED (positive = member was hurting)\n')
print(ab.sort_values(ISOS[-1], ascending=False).to_string(float_format=lambda x: f'{x:+.4f}'))
print('\n|delta| < 0.005 is noise.')

## Cell 12–13 — Does the local number transfer? (the calibrator, and its own audit)

A binormal ROC turns discriminability into MCC:

$$\mathrm{TPR}=1-\Phi(t-d'),\qquad \mathrm{FPR}=1-\Phi(t),\qquad d'=\sqrt{2}\,\Phi^{-1}(\mathrm{AUC})$$

Everything downstream rests on it, so the cell **audits it against the scored
submission before using it**: given our blind `d'` and the recovered prevalence, does it
reproduce the MA-MCC we actually got? On `submission_tdi_best` it did, to 0.0008.

The second table is the part that matters for judging a new model: the **shrinkage** from
out-of-fold `d'` to blind `d'`. That ratio is the TDI analogue of the pIC50
blind-matched validator, and it is why an OOF gain is not a leaderboard gain.

In [ ]:
def d_prime(auc):        return np.sqrt(2) * norm.ppf(np.clip(auc, 1e-6, 1 - 1e-6))
def auc_of(d):           return norm.cdf(d / np.sqrt(2))

def mcc_binormal(d, prev, t):
    tpr, fpr = 1 - norm.cdf(t - d), 1 - norm.cdf(t)
    TP, FN = prev * tpr, prev * (1 - tpr)
    FP, TN = (1 - prev) * fpr, (1 - prev) * (1 - fpr)
    den = np.sqrt((TP + FP) * (TP + FN) * (TN + FP) * (TN + FN))
    return ((TP * TN - FP * FN) / den if den > 0 else 0.0), TP + FP

def mcc_at_rate(d, prev, rate):
    ts = np.linspace(-3, 5, 8000)
    v = [mcc_binormal(d, prev, t) for t in ts]
    i = int(np.argmin([abs(x[1] - rate) for x in v]))
    return v[i][0]

def optimum(d, prev):
    ts = np.linspace(-3, 5, 8000)
    v = [mcc_binormal(d, prev, t) for t in ts]
    i = int(np.argmax([x[0] for x in v]))
    return v[i][1], v[i][0]

BLIND_DPRIME = {'CYP3A4': 1.290, 'CYP2D6': 0.980}     # from submission_tdi_best
BLIND_RATE   = {'CYP3A4': 0.2290, 'CYP2D6': 0.2136}   # EFFECTIVE rates it was scored at

print('AUDIT -- can the model reproduce submission_tdi_best?\n')
print(f"  {'endpoint':8s} {'d-prime':>8s} {'rate':>7s} {'predicted':>10s} {'actual':>8s} {'err':>8s}")
errs = []
for c in ISOS:
    pred = mcc_at_rate(BLIND_DPRIME[c], TEST_PREVALENCE_BY_ISO[c], BLIND_RATE[c])
    errs.append(pred - BEST_MCC[c])
    print(f'  {c:8s} {BLIND_DPRIME[c]:8.3f} {BLIND_RATE[c]:7.4f} {pred:10.4f} '
          f'{BEST_MCC[c]:8.4f} {pred-BEST_MCC[c]:+8.4f}')
print(f"\n  mean |error| {np.mean(np.abs(errs)):.4f} -- "
      f"{'trustworthy for threshold work' if np.mean(np.abs(errs)) < 0.01 else 'DO NOT ACT ON THIS MODEL'}")

print('\n\nSHRINKAGE -- out-of-fold d-prime against blind d-prime\n')
print(f"  {'endpoint':8s} {'OOF AUC':>8s} {'OOF d':>7s} {'blind d':>8s} {'ratio':>7s}")
shrink = {}
for c in ISOS:
    d = oof[c]; y = d['y'].to_numpy().astype(bool)
    a = roc_auc_score(y, blend(d, MEMBERS))
    shrink[c] = BLIND_DPRIME[c] / d_prime(a)
    print(f'  {c:8s} {a:8.4f} {d_prime(a):7.3f} {BLIND_DPRIME[c]:8.3f} {shrink[c]:7.3f}')
print()
print('  READ THIS BEFORE USING THE RATIO. It is only meaningful when the OOF pool above')
print('  is the SAME pool that produced the scored submission the blind d-prime came')
print('  from. Compare `MEMBERS` against the members of submission_tdi_best; if they')
print('  differ, the ratio mixes two models and means nothing.')
print()
print('  A ratio near 1.0 says out-of-fold predicts the leaderboard. A ratio far above')
print('  1.0 says out-of-fold UNDERSTATES the model -- which is not good news, it is a')
print('  warning that the two populations differ enough that local comparisons between')
print('  candidate models are being made on the wrong chemistry. Three explanations fit,')
print('  and they are distinguishable:')
print('    (a) composition -- our folds carry 21.6% CYP2D6 positives against a blind')
print('        7.06%; AUC is prevalence-invariant in expectation, so a large gap means')
print('        the positives themselves differ, not just how many there are;')
print('    (b) the leaderboard scores only the SUBSET with TDI ground truth, and if that')
print('        subset is enriched for easy calls the blind d-prime is flattered;')
print('    (c) the inversion is wrong -- least likely, since it recovered the same')
print('        prevalence to four decimals on every competitor.')

## Cell 14–15 — The threshold

Two different rates, and conflating them has already cost a submission's worth of gain:

* **effective rate** — what the leaderboard sees. It scores only the subset of the 750
  with TDI ground truth for that isoform, and our predicted positives are enriched in it.
* **submitted rate** — what fraction of the file we mark positive.

Measured on `submission_tdi_best`: 0.2107 submitted → 0.2290 seen (CYP3A4), 0.1707 →
0.2136 (CYP2D6). So cut the file at `effective / inflation`, or the gain is thrown away
by overshooting.

The curve is flat near its peak, which is the useful part: it means a cut chosen from a
slightly wrong `d'` costs little, while a cut chosen at the wrong *end* costs a lot.

In [ ]:
print('MCC vs EFFECTIVE cut rate, at our measured d-prime\n')
for c in ISOS:
    d, prev = BLIND_DPRIME[c], TEST_PREVALENCE_BY_ISO[c]
    q, m = optimum(d, prev)
    print(f'--- {c}   d\' {d:.3f}   prevalence {prev:.4f} ---')
    print(f"  {'effective':>10s} {'MCC':>8s} {'submit at':>10s}")
    for rate in (0.05, 0.08, 0.10, 0.14, 0.17, 0.21, 0.25, 0.29, 0.31, 0.35):
        star = '  <- optimum' if abs(rate - q) < 0.021 else ''
        print(f'  {rate:10.3f} {mcc_at_rate(d, prev, rate):8.4f} '
              f'{submitted_rate(c, rate):10.3f}{star}')
    print(f'  argmax: effective {q:.3f} -> MCC {m:.4f}, '
          f'cut the file at {submitted_rate(c, q):.3f}')
    print(f'  last time: effective {BLIND_RATE[c]:.4f} -> {BEST_MCC[c]:.4f} '
          f'(gain available {m - BEST_MCC[c]:+.4f})\n')

print(f'configured OPTIMAL_RATE (effective): {OPTIMAL_RATE}')
print('cut the file at         :',
      {c: round(submitted_rate(c, OPTIMAL_RATE[c]), 4) for c in ISOS})

## Cell 16–17 — Validate the blind predictions themselves

Training being sound does not make the predictions sound. Three checks:

1. **distribution shift** — the blind score distribution against the out-of-fold one.
   Rank-averaged scores are uniform by construction on the training folds, so a blind
   histogram that is *not* roughly uniform means the pool is extrapolating.
2. **realised rate at the chosen cut** — what we actually mark positive.
3. **implied prevalence** — if the score is calibrated at all, the fraction above the cut
   should sit near the recovered blind prevalence. A large gap is a warning, not proof.

In [ ]:
CUT_RATE = {c: submitted_rate(c, OPTIMAL_RATE[c]) for c in ISOS}   # <-- edit to explore

print(f"  {'endpoint':8s} {'OOF mean/sd':>16s} {'blind mean/sd':>16s} "
      f"{'cut':>7s} {'rate':>7s} {'implied eff':>12s}")
for c in ISOS:
    d = oof[c]
    so = blend(d, MEMBERS); so = (so - so.min()) / (so.max() - so.min())
    col = f'{c}_score'
    sb = blend_test[col].to_numpy(float) if col in blend_test.columns else None
    if sb is None:
        print(f'  {c}: no {col} in testscores_{TAG}.csv'); continue
    sbn = (sb - sb.min()) / (sb.max() - sb.min())
    cut = float(np.quantile(sb, 1 - CUT_RATE[c]))
    got = float((sb >= cut).mean())
    print(f'  {c:8s} {so.mean():7.3f}/{so.std():6.3f} {sbn.mean():7.3f}/{sbn.std():6.3f} '
          f'{cut:7.3f} {got:7.3f} {got*RATE_INFLATION.get(c,1.0):12.3f}')

print('\ndecile counts of the blind score (uniform-ish is healthy):')
for c in ISOS:
    col = f'{c}_score'
    if col not in blend_test.columns:
        continue
    sb = blend_test[col].to_numpy(float)
    h = np.histogram(pd.Series(sb).rank(pct=True), bins=10, range=(0, 1))[0]
    print(f'  {c}: {h}')

## Cell 18–19 — Diff against the standing best — **the cell that would have caught `tdi_recut`**

Column agreement is not the right statistic. `tdi_recut` changed 2.1% of labels and lost
0.036 MA-MCC, because a label only changes when its score sits near the threshold, and
those are exactly the rows MCC is most sensitive to.

So this compares the two files *and* asks where the changed rows sit relative to the cut.
Then it states a verdict:

* a **threshold-only** change is predictable by the calibrator to about 0.001 — act on it;
* a **score-column** change is not predictable at all, and needs out-of-fold AUC evidence
  larger than the shrinkage-corrected noise before it earns a slot.

In [ ]:
new = blend_test[[CC.ID_COL]].copy()
for c in ISOS:
    s = blend_test[f'{c}_score'].to_numpy(float)
    new[f'{c}_is_TDI'] = s >= float(np.quantile(s, 1 - CUT_RATE[c]))

if not BASELINE.exists():
    print(f'{BASELINE.name} not found -- cannot diff. This is the check that matters; '
          f'find the file before submitting.')
else:
    old = pd.read_csv(BASELINE); old.columns = [x.strip() for x in old.columns]
    j = new.merge(old, on=CC.ID_COL, suffixes=('_new', '_old'))
    print(f'compared on {len(j)} compounds against {BASELINE.name}\n')
    for c in ISOS:
        a = j[f'{c}_is_TDI_new'].astype(bool).to_numpy()
        b = j[f'{c}_is_TDI_old'].astype(bool).to_numpy()
        ch = a != b
        s = blend_test.set_index(CC.ID_COL).loc[j[CC.ID_COL], f'{c}_score'].to_numpy(float)
        cut = float(np.quantile(blend_test[f'{c}_score'], 1 - CUT_RATE[c]))
        pct = pd.Series(s).rank(pct=True).to_numpy()
        cutpct = 1 - CUT_RATE[c]
        near = np.abs(pct - cutpct) < 0.05
        print(f'{c}: agreement {100*(1-ch.mean()):.1f}%   {ch.sum()} labels change '
              f'({a.sum()} positive now vs {b.sum()} before)')
        if ch.any():
            print(f'   of the changed rows, {100*near[ch].mean():.0f}% sit within 5 '
                  f'percentiles of the cut -- MCC is most sensitive exactly there')
        print(f'   0 -> 1: {int((a & ~b).sum())}    1 -> 0: {int((~a & b).sum())}')

    print('\nVERDICT')
    pred = {c: mcc_at_rate(BLIND_DPRIME[c],
                           TEST_PREVALENCE_BY_ISO[c],
                           CUT_RATE[c] * RATE_INFLATION.get(c, 1.0)) for c in ISOS}
    print(f'  predicted MA-MCC at these cuts, IF d-prime is unchanged: '
          f'{np.mean(list(pred.values())):.4f}   ({ {k: round(v,4) for k,v in pred.items()} })')
    print(f'  standing best                                          : {BEST_MA:.4f}')
    gain = np.mean(list(pred.values())) - BEST_MA
    print(f'  predicted gain {gain:+.4f}')
    print('\n  That prediction is only valid if the SCORES are the same as the scored')
    print('  submission and only the cut moved. If the pool changed, d-prime changed too,')
    print('  and the honest bar is out-of-fold AUC evidence -- shrinkage-corrected --')
    print('  not this number. tdi_recut predicted +0.0087 on the threshold, got it, and')
    print('  still lost 0.036 because the score column had changed underneath it.')

## Cell 20 — Write the submission

Writes only when you set `WRITE = True`, so the diff above has to be read first.

In [ ]:
WRITE = False
OUT = ART / f'submission_{TAG}.csv'

if not WRITE:
    print('WRITE is False -- nothing written. Read Cell 19 first, then set WRITE = True.')
else:
    test = pd.read_csv(REPO / 'data' / 'cyp-challenge-TEST-BLINDED.csv')
    test.columns = [c.strip() for c in test.columns]
    out = test[[CC.SMILES_COL, CC.ID_COL]].copy()
    m = blend_test.set_index(CC.ID_COL)
    for c in ISOS:
        s = out[CC.ID_COL].map(m[f'{c}_score']).to_numpy(float)
        assert np.isfinite(s).all(), f'{c}: blind scores do not cover every test compound'
        out[f'{c}_is_TDI'] = s >= float(np.quantile(s, 1 - CUT_RATE[c]))
        print(f'  {c}: {out[f"{c}_is_TDI"].mean():.4f} of the file marked positive '
              f'(~{out[f"{c}_is_TDI"].mean()*RATE_INFLATION.get(c,1.0):.3f} effective)')
    out.to_csv(OUT, index=False)
    print(f'\nwritten {OUT}')

## Cell 21 — Where the remaining gain is, and how to test a candidate here

CYP3A4 is a solved model with a fixable cut, worth about **+0.009**. CYP2D6 is the whole
rest of the gap: `d'` 0.980 against the leader's 1.392, which is roughly **+0.12 of MCC**
on that endpoint alone. Everything below is aimed at CYP2D6 AUC.

Ranked by the argument for them, not by ease:

1. **The unused CYP1A2 / CYP2C9 TDI labels.** `is_TDI` is only *scored* on CYP3A4 and
   CYP2D6, but `TRAIN_Emax` carries it for all four — 1,413 + 1,285 rows. Time-dependent
   inhibition is mechanism-based: it comes from reactive metabolites and structural
   alerts largely regardless of which isoform bioactivates the compound. The `pooled`
   member already exploits this; the ablation in Cell 11 tells you whether it is
   actually earning its place, and if it is, the question is why it is only *one*
   member rather than the backbone.
2. **Reactive-metabolite alerts as explicit features** (`cypkit/tdi_alerts.py`). TDI has a
   mechanistic basis that ECFP does not encode directly — furans, thiophenes, terminal
   alkynes, anilines, methylenedioxyphenyl. Check the correlation matrix first: if an
   alert-only member correlates below 0.7 with `lgb_clf`, it can contribute.
3. **A different representation.** Every member here is a tree or a kernel over the same
   two feature families. This is the TDI counterpart of the pIC50 finding that two
   ChemProp variants agreeing at 0.85 buy +0.011 — decorrelation is where blends gain,
   and `unimol_kaggle_cyp_tdi_5fold.ipynb` is the cheapest source of a genuinely
   different one.
4. **Class weighting.** CYP2D6 blind prevalence is 0.0706 — the rarest thing in this
   project. Verify in Cell 5 what the out-of-fold prevalence is; if it is much higher,
   the members are being trained on a population the leaderboard does not contain.

**Do not** spend effort on CYP2D6 pIC50 as a TDI feature. It was measured: AUC 0.4409,
*inverted*, and it degrades the blend monotonically up to −0.058 at weight 0.40. The
mechanism is clean — TDI is a shift in potency after preincubation, and a compound that
is already a potent direct inhibitor has no headroom to shift.

### Scaffold for testing a candidate

Fill in `my_score` and this reports whether it belongs in the pool, using the same
criteria as the cells above: standalone AUC, correlation with what we have, and the
change to the blend.

In [ ]:
CAND = 'CYP2D6'          # the endpoint that matters
d = oof[CAND]; y = d['y'].to_numpy().astype(bool)

# ---- your candidate: one score per row of `d`, higher = more likely TDI -------------
my_score = None          # e.g. np.load(...), or a model you fit on d[CC.ID_COL]
# ------------------------------------------------------------------------------------

if my_score is None:
    print('Set `my_score` to a per-row score array of length', len(d))
else:
    my_score = np.asarray(my_score, float)
    assert len(my_score) == len(d), f'expected {len(d)} scores, got {len(my_score)}'
    base = blend(d, MEMBERS)
    print(f'standalone AUC        : {roc_auc_score(y, my_score):.4f}')
    print(f'current blend AUC     : {roc_auc_score(y, base):.4f}')
    print('\ncorrelation with existing members (>0.95 = redundant):')
    for m in MEMBERS:
        print(f'  {m:12s} {spearmanr(my_score, d[m]).statistic:+.3f}')
    print('\nblend AUC with the candidate added, by weight:')
    r_new, r_old = pd.Series(my_score).rank().to_numpy(), pd.Series(base).rank().to_numpy()
    r_old = r_old / r_old.max(); r_new = r_new / r_new.max()
    best = (0.0, roc_auc_score(y, r_old))
    for w in (0.0, 0.1, 0.2, 0.3, 0.4, 0.5):
        a = roc_auc_score(y, (1 - w) * r_old + w * r_new)
        print(f'  w={w:.1f}  AUC {a:.4f}  ({a - best[1]:+.4f})')
    print('\nBar to clear: the shrinkage in Cell 13 means an out-of-fold AUC gain arrives')
    print('at the leaderboard smaller than it looks. A gain under ~0.01 is not a')
    print('submission -- it is a note to keep the member and look for the next one.')